graph based risk aware aircraft colision avoidance using value iteration and policy iteration 

Graph-Risk MDP: Dynamic Programming Based Aircraft Collision Avoidance using OpenSky Encounter Data

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter,defaultdict
import os
import re
import json

In [2]:
df=pd.read_csv('opensky_collision_avoidance_dataset.csv')
df.head()

,time,own_icao24,intruder_icao24,own_latitude,own_longitude,own_geo_altitude_m,own_velocity_ms,own_heading_deg,own_vertical_rate_ms,intruder_latitude,...,intruder_vertical_rate_ms,relative_distance_km,relative_altitude_m,relative_speed_proxy_ms,distance_state,altitude_state,closing_state,mdp_state,recommended_action,reward
0,1781283492,06a081,4d238c,21.5555,114.0651,5844.54,216.48,193.61,6.83,22.1790,...,-2.60,77.029,-5044.44,113.36,far,below,approaching,far_below_approaching,maintain,5
1,1781283492,06a081,789239,21.5555,114.0651,5844.54,216.48,193.61,6.83,22.0067,...,-2.93,50.412,-3954.78,87.95,far,below,approaching,far_below_approaching,maintain,5
2,1781283492,06a081,78923f,21.5555,114.0651,5844.54,216.48,193.61,6.83,22.0112,...,-2.93,58.748,-3512.82,73.93,far,below,approaching,far_below_approaching,maintain,5
3,1781283492,06a081,789285,21.5555,114.0651,5844.54,216.48,193.61,6.83,21.9698,...,-2.60,46.128,-4221.48,89.61,far,below,approaching,far_below_approaching,maintain,5
4,1781283492,3c6717,0101dd,39.9681,116.8209,10066.02,261.74,353.00,-0.33,40.4816,...,7.80,61.726,-6629.40,80.91,far,below,approaching,far_below_approaching,maintain,5


In [3]:
df.shape

(102174, 24)

In [4]:
df.columns 

Index(['time', 'own_icao24', 'intruder_icao24', 'own_latitude',
       'own_longitude', 'own_geo_altitude_m', 'own_velocity_ms',
       'own_heading_deg', 'own_vertical_rate_ms', 'intruder_latitude',
       'intruder_longitude', 'intruder_geo_altitude_m', 'intruder_velocity_ms',
       'intruder_heading_deg', 'intruder_vertical_rate_ms',
       'relative_distance_km', 'relative_altitude_m',
       'relative_speed_proxy_ms', 'distance_state', 'altitude_state',
       'closing_state', 'mdp_state', 'recommended_action', 'reward'],
      dtype='object')

In [5]:
df['mdp_state'].nunique()

18

In [6]:
df['recommended_action'].value_counts()

recommended_action
maintain    100353
climb         1400
descend        421
Name: count, dtype: int64

lets make transitio 

In [7]:
df=df.sort_values(['own_icao24','intruder_icao24','time']).reset_index(drop=True)
df['next_state']=df.groupby(['own_icao24','intruder_icao24'])['mdp_state'].shift(-1)
df=df.dropna(subset=['next_state']).copy()

states=sorted(df['mdp_state'].unique())
actions=sorted(df['recommended_action'].unique())
print(f'number of states: {len(states)}')
print(f'number of actions: {len(actions)}')

number of states: 18
number of actions: 3


MDP probability 

In [8]:
transition_counts=defaultdict(Counter)
reward_values=defaultdict(list)

for _,row in df.iterrows():
    s=row['mdp_state']
    a=row['recommended_action']
    s_next=row['next_state']
    r=row['reward']
    
    transition_counts[(s,a)][s_next]+=1
    reward_values[(s,a)].append(r)
    
p={}
r={}
for key,counter in transition_counts.items():
    total=sum(counter.values())
    p[key]={s_next:count/total for s_next,count in counter.items()}
    
    
for key,rewards in reward_values.items():
    r[key]=np.mean(rewards)

value iteration 

policy dictionary 

In [11]:
gamma=0.99 
theta = 1e-6 
max_iter=1000

v={s:0 for s in states}
for it in range(max_iter):
    delta= 0 
    for s in states:
        q_values=[]
        
        for a in actions:
            if (s,a) not in p :
                q_values.append(-1e9)
                continue
            
            q=0 
            for ns ,prob in p[(s,a)].items():
                reward= r.get((s,a,ns),0)
                q+=prob*(reward + gamma * v[ns])
            q_values.append(q)
            
        best_value=max(q_values)
        delta=max(delta,abs(best_value-v[s]))
        v[s]=best_value
        
    if delta < theta:
        print(f'Value iteration converged after {it+1} iterations.')
        break
value_policy={}
for s in states:
    best_a = None 
    best_q = -1e9
    for a in actions:
        if (s,a) not in p:
            continue 
        q=0 
        for ns,prob in p[(s,a)].items():
            reward=r.get((s,a,ns),0)
            q+=prob*(reward + gamma * v[ns])
        if q > best_q:
            best_q=q
            best_a=a
    value_policy[s]=best_a
value_policy 

Value iteration converged after 1 iterations.


{'far_above_approaching': 'maintain',
 'far_above_stable': 'maintain',
 'far_below_approaching': 'maintain',
 'far_below_stable': 'maintain',
 'far_same_approaching': 'maintain',
 'far_same_stable': 'maintain',
 'medium_above_approaching': 'maintain',
 'medium_above_stable': 'maintain',
 'medium_below_approaching': 'maintain',
 'medium_below_stable': 'maintain',
 'medium_same_approaching': 'climb',
 'medium_same_stable': 'maintain',
 'near_above_approaching': 'climb',
 'near_above_stable': 'climb',
 'near_below_approaching': 'descend',
 'near_below_stable': 'descend',
 'near_same_approaching': 'climb',
 'near_same_stable': 'climb'}

policy iteration 

In [12]:
policy = {s: np.random.choice(actions) for s in states}

def policy_evaluation(policy, gamma=0.90, theta=1e-6):
    V = {s: 0.0 for s in states}

    while True:
        delta = 0

        for s in states:
            a = policy[s]

            if (s, a) not in P:
                continue

            old_v = V[s]
            new_v = 0

            for ns, prob in P[(s, a)].items():
                reward = R.get((s, a, ns), 0)
                new_v += prob * (reward + gamma * V[ns])

            V[s] = new_v
            delta = max(delta, abs(old_v - new_v))

        if delta < theta:
            break

    return V


def policy_improvement(V, policy):
    stable = True
    new_policy = policy.copy()

    for s in states:
        old_action = policy[s]

        best_a = None
        best_q = -1e9

        for a in actions:
            if (s, a) not in P:
                continue

            q = 0
            for ns, prob in P[(s, a)].items():
                reward = R.get((s, a, ns), 0)
                q += prob * (reward + gamma * V[ns])

            if q > best_q:
                best_q = q
                best_a = a

        if best_a is not None:
            new_policy[s] = best_a

        if new_policy[s] != old_action:
            stable = False

    return new_policy, stable


for it in range(100):
    V_pi = policy_evaluation(policy)
    new_policy, stable = policy_improvement(V_pi, policy)

    policy = new_policy

    if stable:
        print("Policy Iteration converged at iteration:", it)
        break

policy_iteration_policy = policy
policy_iteration_policy

NameError: name 'P' is not defined